# Libraries import

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Data import

In [9]:
data = pd.read_csv("student_dropout/data.csv",sep=';')
data.rename(columns=lambda x: x.strip(), inplace=True)

data.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


# Data preprocessing

## Label name transformation

In [10]:
column_mapping = {
    "Nacionality":"Nationality",
    'Previous qualification (grade)': 'Grade of previous qualification',
    'Curricular units 1st sem (credited)': '1st Sem Credited',
    'Curricular units 1st sem (enrolled)': '1st Sem Enrolled',
    'Curricular units 1st sem (evaluations)': '1st Sem Evaluations',
    'Curricular units 1st sem (approved)': '1st Sem Approved',
    'Curricular units 1st sem (grade)': '1st Sem Grade',
    'Curricular units 1st sem (without evaluations)': '1st Sem No Eval',
    'Curricular units 2nd sem (credited)': '2nd Sem Credited',
    'Curricular units 2nd sem (enrolled)': '2nd Sem Enrolled',
    'Curricular units 2nd sem (evaluations)': '2nd Sem Evaluations',
    'Curricular units 2nd sem (approved)': '2nd Sem Approved',
    'Curricular units 2nd sem (grade)': '2nd Sem Grade',
    'Curricular units 2nd sem (without evaluations)': '2nd Sem No Eval'
}

data.rename(columns=column_mapping, inplace=True)

data.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Grade of previous qualification,Nationality,Mother's qualification,Father's qualification,...,2nd Sem Credited,2nd Sem Enrolled,2nd Sem Evaluations,2nd Sem Approved,2nd Sem Grade,2nd Sem No Eval,Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## Categorical variable transformation

Lowering the number of categories for some categorical variables

In [11]:
def map_marital_status(category):
    return category if category == 1 else 0

def map_application_order(category):
    return category if category in [1, 2, 3] else 0

def map_previous_qualification(category):
    return category if category in [1, 39, 19, 3] else 0

def map_nationality(category):
    return category if category==1 else 0

def map_mothers_qualification(category):
    return category if category in [1, 37, 19, 38, 3] else 0

def map_fathers_qualification(category):
    return category if category in [37, 19, 1, 38] else 0

def map_mothers_occupation(category):
    return category if category in [9, 4, 5, 3] else 0

def map_fathers_occupation(category):
    return category if category in [9, 7, 4, 5, 3, 8] else 0

def map_application_mode(category):
    return category if category in [1,17,39] else 0

def categorize_age(age):
    if 18 <= age <= 23:
        return 1
    elif 24 <= age <= 30:
        return 2
    elif 31 <= age <= 40:
        return 3
    else:
        return 4

In [12]:
data["Marital status"] = data["Marital status"].apply(map_marital_status)
data["Application order"] = data["Application order"].apply(map_application_order)
data["Previous qualification"] = data["Previous qualification"].apply(map_previous_qualification)
data["Nationality"] = data["Nationality"].apply(map_nationality)
data["Mother's qualification"] = data["Mother's qualification"].apply(map_mothers_qualification)
data["Father's qualification"] = data["Father's qualification"].apply(map_fathers_qualification)
data["Mother's occupation"] = data["Mother's occupation"].apply(map_mothers_occupation)
data["Father's occupation"] = data["Father's occupation"].apply(map_fathers_occupation)
data["Application mode"] = data["Application mode"].apply(map_application_mode)
data["Age at enrollment"] = data["Age at enrollment"].apply(categorize_age)

# Train test split

In [13]:
from sklearn.model_selection import train_test_split

X = data.drop('Target', axis=1)  # Features
y = data['Target']               # Target variable

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Detecting Outliers

In [15]:
excluded_columns = [
    "Marital status", "Application mode", "Application order", "Course",
    "Daytime/evening attendance", "Previous qualification", "Nationality",
    "Mother's qualification", "Father's qualification", "Mother's occupation",
    "Father's occupation", "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder",
    "Age at enrollment", "International"
]

# Identify columns not in the excluded list and are numerical
columns_to_analyze = [col for col in X_train.columns if col not in excluded_columns and pd.api.types.is_numeric_dtype(X_train[col])]

# Function to calculate IQR and identify outliers
def detect_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return series[(series < lower_bound) | (series > upper_bound)]

# Applying the outlier detection
outliers = {col: detect_outliers(X_train[col]) for col in columns_to_analyze}

# Displaying the outliers
print(outliers)

{'Grade of previous qualification': 2431    172.0
3292    170.0
2439    170.0
1872    170.0
2176    167.0
1283    168.0
660     180.0
2793    168.0
2221     99.0
702     190.0
399      96.0
4394    168.0
2751    170.0
3024    169.0
349      96.0
1443    170.0
3861    167.0
2457    166.0
2564    172.0
142      99.0
3214    170.0
394     166.0
1294    168.0
2129    182.0
1591    172.0
3961    168.0
2375     95.0
974     170.0
53      167.0
2990    172.0
696     165.0
2262    178.0
341     188.0
1120    170.0
1046    175.0
1888    184.4
3813    180.0
1254    170.0
1215    176.0
3144    165.0
2128    170.0
577     172.0
1555     97.0
2214    170.0
1008    165.0
524     180.0
2455    170.0
2901    170.0
2205    180.0
2557    165.0
2511    190.0
3943    180.0
3152    177.0
Name: Grade of previous qualification, dtype: float64, 'Admission grade': 2011    160.0
4208    160.4
3711    160.0
2664    168.2
678     163.5
        ...  
1733    172.0
3457    160.0
1853    160.0
1597    161.2
189     

# Scaling the non categorical variables

In [16]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

excluded_columns = [
    "Marital status", "Application mode", "Application order", "Course",
    "Daytime/evening attendance", "Previous qualification", "Nationality",
    "Mother's qualification", "Father's qualification", "Mother's occupation",
    "Father's occupation", "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder",
    "Age at enrollment", "International", "Target"
]

columns_to_scale = [col for col in X_train.columns if col not in excluded_columns]

scaler = StandardScaler()

X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

X_train.head()


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Grade of previous qualification,Nationality,Mother's qualification,Father's qualification,...,1st Sem No Eval,2nd Sem Credited,2nd Sem Enrolled,2nd Sem Evaluations,2nd Sem Approved,2nd Sem Grade,2nd Sem No Eval,Unemployment rate,Inflation rate,GDP
3691,1,17,0,9119,1,1,-0.191029,1,19,1,...,-0.193039,-0.284744,-0.548424,1.507892,-1.152821,0.142417,-0.198343,-0.999750,0.116919,1.537012
1844,1,1,0,9500,1,1,0.035727,1,1,1,...,-0.193039,0.248247,0.822709,0.750767,1.187594,0.404166,-0.198343,0.310565,-0.535559,0.781730
613,1,17,3,9147,1,1,-1.098051,1,1,19,...,-0.193039,-0.284744,-0.548424,-0.258733,0.184559,0.565489,-0.198343,0.310565,-0.535559,0.781730
2974,1,1,1,171,1,1,-0.644540,1,1,19,...,-0.193039,-0.284744,-2.833645,-2.025359,-1.487166,-1.972944,-0.198343,0.872129,-1.115540,0.342612
1611,1,0,0,9085,1,1,0.111312,1,19,37,...,2.751050,-0.284744,-0.548424,-0.258733,0.184559,0.411645,-0.198343,-0.812562,-1.478028,-1.374337


# Encoding the categorical variables

In [19]:
columns_to_encode = [
    "Marital status", "Application mode", "Application order", "Course",
    "Daytime/evening attendance", "Previous qualification", "Nationality",
    "Mother's qualification", "Father's qualification", "Mother's occupation",
    "Father's occupation", "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder",
    "Age at enrollment", "International"
]
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

X_train_encoded = encoder.fit_transform(X_train[columns_to_encode])
X_test_encoded = encoder.transform(X_test[columns_to_encode])

X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out(columns_to_encode))
X_test_encoded = pd.DataFrame(X_test_encoded, columns=encoder.get_feature_names_out(columns_to_encode))

X_train = X_train.drop(columns=columns_to_encode).reset_index(drop=True)
X_test = X_test.drop(columns=columns_to_encode).reset_index(drop=True)

X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

X_train.head()

/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


,Grade of previous qualification,Admission grade,1st Sem Credited,1st Sem Enrolled,1st Sem Evaluations,1st Sem Approved,1st Sem Grade,1st Sem No Eval,2nd Sem Credited,2nd Sem Enrolled,...,Gender_0,Gender_1,Scholarship holder_0,Scholarship holder_1,Age at enrollment_1,Age at enrollment_2,Age at enrollment_3,Age at enrollment_4,International_0,International_1
0,-0.191029,-0.605736,-0.301579,-0.500859,0.648623,-1.206052,-0.132999,-0.193039,-0.284744,-0.548424,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,0.035727,-0.494742,0.565356,0.311643,-0.060882,0.747185,0.503392,-0.193039,0.248247,0.822709,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,-1.098051,-0.612673,-0.301579,-0.500859,0.648623,-0.229434,0.073957,-0.193039,-0.284744,-0.548424,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,-0.644540,-0.897096,-0.301579,-2.532115,-1.952897,-1.531592,-2.202564,-0.193039,-0.284744,-2.833645,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.111312,-0.203382,-0.301579,-0.500859,0.648623,0.096106,0.487870,2.751050,-0.284744,-0.548424,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [21]:
replacement_dict = {
    "Dropout": 0,
    "Enrolled": 1,
    "Graduate": 2
}

y_train = y_train.replace(replacement_dict)
y_test = y_test.replace(replacement_dict)


print(y_train.head())

3691    0
1844    2
613     1
2974    0
1611    1
Name: Target, dtype: int64


In [22]:
X_train.head()

,Grade of previous qualification,Admission grade,1st Sem Credited,1st Sem Enrolled,1st Sem Evaluations,1st Sem Approved,1st Sem Grade,1st Sem No Eval,2nd Sem Credited,2nd Sem Enrolled,...,Gender_0,Gender_1,Scholarship holder_0,Scholarship holder_1,Age at enrollment_1,Age at enrollment_2,Age at enrollment_3,Age at enrollment_4,International_0,International_1
0,-0.191029,-0.605736,-0.301579,-0.500859,0.648623,-1.206052,-0.132999,-0.193039,-0.284744,-0.548424,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,0.035727,-0.494742,0.565356,0.311643,-0.060882,0.747185,0.503392,-0.193039,0.248247,0.822709,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,-1.098051,-0.612673,-0.301579,-0.500859,0.648623,-0.229434,0.073957,-0.193039,-0.284744,-0.548424,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,-0.644540,-0.897096,-0.301579,-2.532115,-1.952897,-1.531592,-2.202564,-0.193039,-0.284744,-2.833645,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.111312,-0.203382,-0.301579,-0.500859,0.648623,0.096106,0.487870,2.751050,-0.284744,-0.548424,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [23]:
# save datasets
X_train.to_csv('student_dropout_preprocessed/X_train.csv', index=False)
X_test.to_csv('student_dropout_preprocessed/X_validation.csv', index=False)

y_train.to_csv('student_dropout_preprocessed/y_train.csv', index=False)
y_test.to_csv('student_dropout_preprocessed/y_validation.csv', index=False)

In [24]:
X_train.shape

(3096, 94)